In [4]:
pip install pyrosm geopandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 9.9 MB/s  0:00:02m eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
from pyrosm import OSM
from scipy.spatial import cKDTree

PBF_FILE = "india-260305.osm.pbf"
CLUSTER_FILE = "Stoppage Intelligence Database - Sheet1"
POI_CACHE = "india_pois.csv"
OUTPUT_FILE = "cluster_nearby_places.csv"

EARTH_RADIUS = 6371
SEARCH_RADIUS_KM = 2


# ----------------------------
# Load or Extract POIs
# ----------------------------

def load_pois():

    try:
        pois = pd.read_csv(POI_CACHE)
        print("Loaded cached POIs")

    except:

        print("Extracting POIs from OSM PBF (first run may take a few minutes)")

        osm = OSM(PBF_FILE)

        pois = osm.get_pois()

        pois = pois[
         (pois["amenity"].isin([
                "fuel",
                "restaurant",
                "fast_food",
                "parking",
                "charging_station"
         ]))
            |
         (pois["highway"].isin([
                "services",
                "rest_area",
                "truck_stop"
         ]))
            |
         (pois["barrier"] == "toll_booth")
            |
         (pois["landuse"] == "industrial")
            |
         (pois["building"] == "warehouse")
        ]
        
        pois = pois[[
            "name",
            "amenity",
            "highway",
            "barrier",
            "landuse",
            "building",
            "lat",
            "lon"
        ]]

        pois = pois.dropna(subset=["name"])

        pois.to_csv(POI_CACHE, index=False)

        print("Saved POIs to cache")

    return pois


# ----------------------------
# Determine place type
# ----------------------------

def determine_place_type(row):

    if pd.notna(row["amenity"]):
        return row["amenity"]

    if pd.notna(row["shop"]):
        return row["shop"]

    if pd.notna(row["tourism"]):
        return row["tourism"]

    return "other"


# ----------------------------
# Build spatial index
# ----------------------------

def build_index(pois):

    coords = np.radians(pois[["lat", "lon"]].values)

    tree = cKDTree(coords)

    return tree


# ----------------------------
# Query nearby places
# ----------------------------

def query_places(tree, pois, lat, lon):

    point = np.radians([lat, lon])

    radius = SEARCH_RADIUS_KM / EARTH_RADIUS

    idx = tree.query_ball_point(point, r=radius)

    return pois.iloc[idx]


# ----------------------------
# Main pipeline
# ----------------------------

def main():

    pois = load_pois()

    pois["place_type"] = pois.apply(determine_place_type, axis=1)

    tree = build_index(pois)

    clusters = pd.read_csv(CLUSTER_FILE)

    results = []

    for _, cluster in clusters.iterrows():

        cid = cluster["cluster_id"]
        lat = cluster["cluster_lat"]
        lon = cluster["cluster_lon"]

        nearby = query_places(tree, pois, lat, lon)

        for _, place in nearby.iterrows():

            results.append({
                "cluster_id": cid,
                "cluster_lat": lat,
                "cluster_lon": lon,
                "place_name": place["name"],
                "place_type": place["place_type"],
                "place_lat": place["lat"],
                "place_lon": place["lon"]
            })

    df = pd.DataFrame(results)

    df.to_csv(OUTPUT_FILE, index=False)

    print("Saved results to:", OUTPUT_FILE)


if __name__ == "__main__":
    main()

Extracting POIs from OSM PBF (first run may take a few minutes)


In [1]:
top

NameError: name 'top' is not defined

In [2]:
top

NameError: name 'top' is not defined

In [3]:
osmium tags-filter india-latest.osm.pbf \
n/amenity=fuel,restaurant,fast_food,parking \
n/highway=services,rest_area,truck_stop \
n/barrier=toll_booth \
n/landuse=industrial \
-o logistics_pois.osm.pbf

SyntaxError: invalid syntax (191783851.py, line 1)